#Crédito Inclusivo (grupo Kurumin):

Este notebook junta, em ordem, todas as partes do grupo. Rode as células de cima pra baixo.

| Parte | O que faz | Arquivo em `partes/` |
|---|---|---|
| 1. Base do Banco Central (SCR) | Baixa e arruma os dados do SCR | `01_base_download_extracao`, `02_base_leitura_tratamento` |
| 2. Inadimplência alta vs recuperação rápida | Gráficos por UF e por região | `03_base_inadimplencia_alta_vs_recuperacao_rapida` |
| 3. Dívida saudável vs crítica | Gráficos por UF e por região | `04_base_divida_saudavel_vs_critica` |
| 4. Taxa de juros | Compara as instituições financeiras | `05_taxa_juros` |
| 5. Apostas (Bets) | Depósitos e apostadores por mês | `06_bets` |
| 6. IBGE | População e renda por estado | `07_ibge_populacao_renda` |

---
# Análise de Risco de Crédito

   1. Baixa os dados abertos do Banco Central (SCR), um .zip por ano.
   2. Abre os .zip e lê os .csv de dentro.
   3. Deixa só o que interessa (Pessoa Física) e arruma as colunas.
   4. Faz as duas análises que já estão prontas:
        - Inadimplência alta vs recuperação rápida
        - Dívida saudável vs crítica
      Cada uma desenha um gráfico no Colab (pra gente conferir os números).

---

Bibliotecas


In [ ]:
import pandas as pd
import requests
import zipfile
import os
import time
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor as tpe
from tqdm.auto import tqdm

---

Configurações


In [ ]:
anos = list(range(2024, 2026 + 1))

pasta_ext = "/content/dados_scr_extraidos"

os.makedirs(pasta_ext, exist_ok=True)

---

Download e extração dos arquivos do Banco Central


In [ ]:
def baixar_zip(ano):
    url = f"https://www.bcb.gov.br/pda/desig/scrdata_{ano}.zip"

    caminho = f"/content/scrdata_{ano}.zip"

    tentativas = 5

    for tentativa in range(1, tentativas + 1):
        try:
            resposta = requests.get(url, stream=True)
            resposta.raise_for_status()

            with open(caminho, "wb") as arquivo:
                for pedaco in resposta.iter_content(chunk_size=1024 * 1024):
                    arquivo.write(pedaco)

            return caminho

        except Exception as erro:
            print(f"Ano {ano}: tentativa {tentativa} de {tentativas} falhou ({erro}).")
            if tentativa < tentativas:
                time.sleep(3)

    print(f"Não foi possível baixar {ano}. Pulando este ano.")
    return None

In [ ]:
with tpe(max_workers=3) as executor:
    resultados = list(tqdm(executor.map(baixar_zip, anos), total=len(anos), desc="Baixando anos"))

zips_baixados = [caminho for caminho in resultados if caminho is not None]
print("Anos baixados com sucesso:", zips_baixados)

for caminho_zip in tqdm(zips_baixados, desc="Extraindo arquivos ZIP"):
    with zipfile.ZipFile(caminho_zip, "r") as zip_aberto:
        zip_aberto.extractall(pasta_ext)

arquivos_csv = [nome for nome in os.listdir(pasta_ext) if nome.endswith(".csv")]
print("Total de arquivos CSV extraídos (todos os anos):", len(arquivos_csv))

---

Leitura dos CSVs


In [ ]:
colunas = [
    "data_base",
    "uf",
    "cliente",
    "porte",
    "carteira_a_vencer",
    "carteira_inadimplencia",
    "a_vencer_ate_90_dias",
    "carteira_ativa",
    "modalidade",
    "submodalidade",
    "ativo_problematico",
]

lista_portes_vulneraveis = ["Sem rendimento", "Até 1 salário mínimo", "Mais de 1 a 2 salários mínimos"]

tabelas_mensais = []

for nome_arquivo in tqdm(arquivos_csv, desc="Lendo arquivos CSV"):
    caminho = os.path.join(pasta_ext, nome_arquivo)

    leitor = pd.read_csv(
        caminho, sep=";", decimal=",", usecols=colunas,
        encoding="utf-8-sig", low_memory=False, chunksize=200_000
    )

    for pedaco in leitor:
        tabelas_mensais.append(pedaco)

tabela_credito = pd.concat(tabelas_mensais, ignore_index=True)
print("Formato da tabela (linhas, colunas):", tabela_credito.shape)

---

Tratamento dos dados


In [ ]:
tabela_pf = tabela_credito[tabela_credito["cliente"] == "PF"].copy()
print("Linhas de Pessoa Física:", tabela_pf.shape[0])

In [ ]:
e_vulneravel = tabela_pf["porte"].isin(lista_portes_vulneraveis)
tabela_pf["publico"] = e_vulneravel.map({True: "vulneravel", False: "demais"})

In [ ]:
mapa_regiao = {
    "AC": "Norte", "AP": "Norte", "AM": "Norte", "PA": "Norte",
    "RO": "Norte", "RR": "Norte", "TO": "Norte",
    "AL": "Nordeste", "BA": "Nordeste", "CE": "Nordeste", "MA": "Nordeste",
    "PB": "Nordeste", "PE": "Nordeste", "PI": "Nordeste", "RN": "Nordeste", "SE": "Nordeste",
    "DF": "Centro-Oeste", "GO": "Centro-Oeste", "MT": "Centro-Oeste", "MS": "Centro-Oeste",
    "ES": "Sudeste", "MG": "Sudeste", "RJ": "Sudeste", "SP": "Sudeste",
    "PR": "Sul", "RS": "Sul", "SC": "Sul"
}

lista_norte_nordeste = [uf for uf, regiao in mapa_regiao.items() if regiao in ["Norte", "Nordeste"]]

menu_regioes = ["Norte + Nordeste", "Norte", "Nordeste", "Centro-Oeste", "Sudeste", "Sul"]

In [ ]:
def escolher_regiao():

    print("Escolha a região:")
    for numero, nome in enumerate(menu_regioes):
        print(f"  {numero}. {nome}")

    while True:
        texto = input("Digite o número (Enter = 0): ").strip()
        if texto == "":
            texto = "0"
        if texto.isdigit() and int(texto) < len(menu_regioes):
            break
        print(f"'{texto}' não é uma opção. Digite um número de 0 a {len(menu_regioes) - 1}.")

    nome = menu_regioes[int(texto)]

    if nome == "Norte + Nordeste":
        ufs = lista_norte_nordeste
    else:
        ufs = [uf for uf, regiao in mapa_regiao.items() if regiao == nome]
    return nome, ufs

---

Inadimplência alta vs recuperação rápida, por UF


In [ ]:
tabela_inadimplencia = tabela_pf.groupby("uf").agg(
    inadimplencia=("carteira_inadimplencia", "sum"),
    a_vencer_90d=("a_vencer_ate_90_dias", "sum")
).reset_index()

tabela_inadimplencia["inadimplencia_bi"] = tabela_inadimplencia["inadimplencia"] / 1e9
tabela_inadimplencia["a_vencer_90d_bi"] = tabela_inadimplencia["a_vencer_90d"] / 1e9

regiao_inadimplencia, ufs_inadimplencia = escolher_regiao()

grafico_inadimplencia = tabela_inadimplencia[tabela_inadimplencia["uf"].isin(ufs_inadimplencia)]
grafico_inadimplencia = grafico_inadimplencia.sort_values(by="inadimplencia_bi", ascending=False)

plt.style.use("ggplot")

eixo = grafico_inadimplencia.set_index("uf")[["inadimplencia_bi", "a_vencer_90d_bi"]].plot(
    kind="bar", figsize=(12, 6), color=["#ef4444", "#4f46e5"], width=0.6
)

for barra in eixo.patches:
    altura = barra.get_height()
    eixo.annotate(f"{altura:.1f}B", (barra.get_x() + barra.get_width() / 2, altura),
                  ha="center", va="bottom", xytext=(0, 3), textcoords="offset points",
                  fontweight="bold", fontsize=7)

plt.title("Inadimplência Alta vs Recuperação Rápida, por UF (R$ bilhões)\nBase SCR - Banco Central do Brasil",
          fontweight="bold", fontsize=12)
plt.xlabel(f"Região {regiao_inadimplencia}", fontweight="bold")
plt.ylabel("R$ bilhões", fontweight="bold")
plt.xticks(rotation=0)
plt.legend(["Inadimplência (alta)", "A vencer em 90 dias (recuperação rápida)"])
plt.ylim(0, grafico_inadimplencia[["inadimplencia_bi", "a_vencer_90d_bi"]].values.max() * 1.15)
plt.savefig("/content/grafico_inadimplencia_uf.png", bbox_inches="tight")
plt.show()

---

Inadimplência alta vs recuperação rápida, por região


In [ ]:
tabela_inadimplencia["regiao"] = tabela_inadimplencia["uf"].map(mapa_regiao)

inadimplencia_regiao = tabela_inadimplencia.groupby("regiao").agg(
    inadimplencia_bi=("inadimplencia_bi", "sum"),
    a_vencer_90d_bi=("a_vencer_90d_bi", "sum")
).reset_index()

eixo = inadimplencia_regiao.set_index("regiao")[["inadimplencia_bi", "a_vencer_90d_bi"]].plot(
    kind="bar", figsize=(8, 5), color=["#ef4444", "#4f46e5"], width=0.6
)
for barra in eixo.patches:
    altura = barra.get_height()
    eixo.annotate(f"{altura:.1f}B", (barra.get_x() + barra.get_width() / 2, altura),
                  ha="center", va="bottom", xytext=(0, 3), textcoords="offset points",
                  fontweight="bold", fontsize=8)

plt.title("Inadimplência Alta vs Recuperação Rápida, por Região (R$ bilhões)", fontweight="bold", fontsize=12)
plt.xlabel("Região", fontweight="bold")
plt.ylabel("R$ bilhões", fontweight="bold")
plt.xticks(rotation=0)
plt.legend(["Inadimplência (alta)", "A vencer em 90 dias (recuperação rápida)"])
plt.ylim(0, inadimplencia_regiao[["inadimplencia_bi", "a_vencer_90d_bi"]].values.max() * 1.15)
plt.savefig("/content/grafico_inadimplencia_regiao.png", bbox_inches="tight")
plt.show()

---

% de dívida saudável vs crítica, por UF


In [ ]:
submod_saudaveis = [
    "Aquisição de bens - veículos automotores",
    "Aquisição de bens com interveniência - veículos autom.",
    "Arrendamento financeiro de veículos automotores",
    "Arrendamento financeiro exceto veículos automotores e imóveis",
    "Financiamento de projeto",
    "Financiamento habitacional - SFH",
    "Financiamento habitacional - exceto SFH",
    "Financiamento imobiliário - empreendim, exceto habitac.",
    "Financiamentos agroindustriais",
    "Home Equity"
]

submod_criticas = [
    "Antecipação de fatura de cartão de crédito",
    "Aquisição de bens com interveniência - outros bens",
    "Cartão de crédito - compra à vista e parcelado lojista",
    "Cartão de crédito - compra, fatura parcelada ou saque financiado pela instituição financeira emitente do cartão",
    "Cheque especial",
    "Crédito pessoal - com consignação em folha de pagam.",
    "Crédito pessoal - sem consignação em folha de pagam.",
    "Crédito rotativo vinculado a cartão de crédito",
    "Microcrédito",
    "Outros com característica de crédito",
    "Outros empréstimos",
    "Outros financiamentos"
]

submod_limpa = tabela_pf["submodalidade"].str.strip().str.replace("\x96", "-", regex=False)

tabela_pf["carteira_saudavel"] = tabela_pf["carteira_ativa"].where(submod_limpa.isin(submod_saudaveis), 0)
tabela_pf["carteira_critica"] = tabela_pf["carteira_ativa"].where(submod_limpa.isin(submod_criticas), 0)

tabela_divida = tabela_pf.groupby("uf").agg(
    carteira_saudavel=("carteira_saudavel", "sum"),
    carteira_critica=("carteira_critica", "sum"),
    carteira_ativa_total=("carteira_ativa", "sum")
).reset_index()

tabela_divida["percentual_saudavel"] = (tabela_divida["carteira_saudavel"] / tabela_divida["carteira_ativa_total"]) * 100
tabela_divida["percentual_critica"] = (tabela_divida["carteira_critica"] / tabela_divida["carteira_ativa_total"]) * 100

tabela_divida["percentual_outra"] = 100 - tabela_divida["percentual_saudavel"] - tabela_divida["percentual_critica"]

regiao_divida, ufs_divida = escolher_regiao()
grafico_divida = tabela_divida[tabela_divida["uf"].isin(ufs_divida)].sort_values(by="percentual_critica")

eixo = grafico_divida.set_index("uf")[["percentual_saudavel", "percentual_critica"]].plot(
    kind="bar", figsize=(12, 6), color=["#10b981", "#ef4444"], width=0.6, legend=False
)

for barra in eixo.patches:
    altura = barra.get_height()
    if altura > 1:
        eixo.annotate(f"{altura:.0f}%", (barra.get_x() + barra.get_width() / 2, barra.get_y() + altura / 2),
                      ha="center", va="center", fontweight="bold", fontsize=8, color="white")

plt.title("% de Dívida Saudável vs Crítica, por UF\nBase SCR - Banco Central do Brasil", fontweight="bold", fontsize=12)
plt.xlabel(f"Região {regiao_divida}", fontweight="bold")
plt.ylabel("% da carteira ativa", fontweight="bold")
plt.xticks(rotation=0)
plt.legend(["Saudável", "Crítica"])
plt.ylim(0, 100)
plt.savefig("/content/grafico_divida_uf.png", bbox_inches="tight")
plt.show()

---

% de dívida saudável vs crítica, por região


In [ ]:
tabela_divida["regiao"] = tabela_divida["uf"].map(mapa_regiao)
divida_regiao = tabela_divida.groupby("regiao").agg(
    carteira_saudavel=("carteira_saudavel", "sum"),
    carteira_critica=("carteira_critica", "sum"),
    carteira_ativa_total=("carteira_ativa_total", "sum")
).reset_index()

divida_regiao["percentual_saudavel"] = (divida_regiao["carteira_saudavel"] / divida_regiao["carteira_ativa_total"]) * 100
divida_regiao["percentual_critica"] = (divida_regiao["carteira_critica"] / divida_regiao["carteira_ativa_total"]) * 100
divida_regiao = divida_regiao.sort_values(by="percentual_critica")

eixo = divida_regiao.set_index("regiao")[["percentual_saudavel", "percentual_critica"]].plot(
    kind="bar", figsize=(8, 5), color=["#10b981", "#ef4444"], width=0.6, legend=False
)
for barra in eixo.patches:
    altura = barra.get_height()
    if altura > 1:
        eixo.annotate(f"{altura:.0f}%", (barra.get_x() + barra.get_width() / 2, barra.get_y() + altura / 2),
                      ha="center", va="center", fontweight="bold", fontsize=9, color="white")

plt.title("% de Dívida Saudável vs Crítica, por Região", fontweight="bold", fontsize=12)
plt.xlabel("Região", fontweight="bold")
plt.ylabel("% da carteira ativa", fontweight="bold")
plt.xticks(rotation=0)
plt.legend(["Saudável", "Crítica"])
plt.ylim(0, 100)
plt.savefig("/content/grafico_divida_regiao.png", bbox_inches="tight")
plt.show()

---
# Taxa de juros das maiores instituições financeiras

**O que este notebook faz**
1. Baixa do Banco Central (API Olinda) as taxas de juros mensais.
2. Deixa só o financiamento imobiliário "com taxas de mercado - Pós-fixado referenciado em TR" e só as 8 instituições que o card pede.
3. Mostra dois gráficos:
   - barras: comparação das taxas no mês mais recente
   - linhas: evolução mês a mês de cada instituição (desde jan/2024)

**Por que financiamento imobiliário pós-fixado em TR?**

A API traz vários tipos de financiamento imobiliário (com taxas de mercado ou reguladas; prefixado, pós-fixado em TR, pós-fixado em IPCA...). Pra comparar os bancos de forma justa, a gente olha o MESMO produto em todos.

TR é a Taxa Referencial, um índice calculado pelo Banco Central. Nesse tipo de financiamento a dívida é corrigida pela TR e o banco cobra os juros por cima. Se misturássemos produtos diferentes, um banco poderia parecer "mais caro" só porque a gente pegou um produto mais caro dele.

---
Configurações da taxa de juros

In [ ]:
url_juros = "https://olinda.bcb.gov.br/olinda/servico/taxaJuros/versao/v2/odata/TaxasJurosMensalPorMes"

colunas_api = "Mes,Modalidade,InstituicaoFinanceira,TaxaJurosAoMes,anoMes"

parte_modalidade = "Pós-fixado referenciado em TR"

parte_tipo = "taxas de mercado"

instituicoes = [
    "BANCO SICOOB S.A.",
    "BANCO COOPERATIVO SICREDI",
    "ITAÚ UNIBANCO S.A.",
    "CAIXA ECONOMICA FEDERAL",
    "BCO SANTANDER (BRASIL) S.A.",
    "BCO BRADESCO S.A.",
    "BANCO INTER",
    "BCO DO BRASIL S.A.",
]

mes_inicial = "2024-01"

---
Download das taxas de juros

In [ ]:
tamanho_pagina = 5000
registros = []
inicio = 0

for numero_pagina in range(400):
    resposta = requests.get(url_juros, params={
        "$top": tamanho_pagina,
        "$skip": inicio,
        "$format": "json",
        "$select": colunas_api,
    })

    if resposta.status_code == 403:
        raise RuntimeError("O Banco Central recusou o pedido (erro 403). Desconecte e exclua o ambiente de execução do Colab, reconecte e rode de novo.")

    resposta.raise_for_status()

    pagina = resposta.json().get("value", [])

    if len(pagina) == 0:
        break

    registros += pagina
    inicio += len(pagina)
    print(f"Linhas baixadas até agora: {len(registros)}")

tabela_juros_bruta = pd.DataFrame(registros)
print("Formato da tabela baixada (linhas, colunas):", tabela_juros_bruta.shape)
print("Período baixado:", tabela_juros_bruta["anoMes"].min(), "até", tabela_juros_bruta["anoMes"].max())

---
Tratamento dos dados

In [ ]:
tabela_juros = tabela_juros_bruta.copy()

e_tr = tabela_juros["Modalidade"].str.contains(parte_modalidade, case=False, na=False)
e_imobiliario = tabela_juros["Modalidade"].str.contains("imobili", case=False, na=False)
e_mercado = tabela_juros["Modalidade"].str.contains(parte_tipo, case=False, na=False)
tabela_juros = tabela_juros[e_tr & e_imobiliario & e_mercado].copy()

print("Modalidades encontradas:")
print(tabela_juros["Modalidade"].value_counts())

def limpar_nome(nomes):
    return (nomes.str.upper()
                 .str.replace(r"[^\w\s]", "", regex=True)
                 .str.replace(r"\s+", " ", regex=True)
                 .str.strip())

nomes_limpos = limpar_nome(pd.Series(instituicoes))
nome_bonito = dict(zip(nomes_limpos, instituicoes))

tabela_juros["instituicao_limpa"] = limpar_nome(tabela_juros["InstituicaoFinanceira"])
tabela_juros = tabela_juros[tabela_juros["instituicao_limpa"].isin(nomes_limpos)].copy()
tabela_juros["instituicao"] = tabela_juros["instituicao_limpa"].map(nome_bonito)

tabela_juros["taxa"] = pd.to_numeric(tabela_juros["TaxaJurosAoMes"], errors="coerce")
tabela_juros = tabela_juros.dropna(subset=["taxa"])

tabela_juros = tabela_juros[tabela_juros["anoMes"].astype(str) >= mes_inicial]

repetidas = tabela_juros.duplicated(["anoMes", "instituicao"]).sum()
if repetidas > 0:
    print(f"ATENÇÃO: {repetidas} linhas repetidas (mesma instituição no mesmo mês). Usando a média delas.")

tabela_juros = tabela_juros.groupby(["anoMes", "Mes", "instituicao"], as_index=False)["taxa"].mean()

if tabela_juros.empty:
    raise RuntimeError("Nenhuma linha sobrou depois dos filtros. Confira a modalidade e os nomes das instituições no bloco 1.")

for nome in instituicoes:
    if nome not in tabela_juros["instituicao"].values:
        print(f"ATENÇÃO: não encontrei '{nome}' nos dados. Confira o nome escrito na lista.")

meses = sorted(tabela_juros["anoMes"].unique())
print("Linhas tratadas:", tabela_juros.shape[0])
print(f"Meses: {len(meses)} (de {meses[0]} até {meses[-1]})")
print("Linhas se TODAS as instituições tivessem taxa em TODOS os meses:", len(meses) * len(instituicoes))
print("Se tiver menos linhas que isso, a API não trouxe taxa de alguma instituição em algum mês (vira buraco no gráfico de linhas).")
display(tabela_juros.head(10))

---
Gráfico de barras: taxas no mês mais recente



In [ ]:
ultimo_mes = tabela_juros["anoMes"].max()
nome_ultimo_mes = tabela_juros.loc[tabela_juros["anoMes"] == ultimo_mes, "Mes"].iloc[0]

grafico_barras = tabela_juros[tabela_juros["anoMes"] == ultimo_mes].sort_values(by="taxa", ascending=False)

plt.style.use("ggplot")

eixo = grafico_barras.set_index("instituicao")["taxa"].plot(
    kind="bar", figsize=(12, 6), color="#4f46e5", width=0.6
)

for barra in eixo.patches:
    altura = barra.get_height()
    eixo.annotate(f"{altura:.2f}%",
                  (barra.get_x() + barra.get_width() / 2, altura),
                  ha="center", va="bottom",
                  xytext=(0, 3), textcoords="offset points",
                  fontweight="bold", fontsize=9)

plt.title(f"Taxa de Juros ao Mês por Instituição - Financiamento Imobiliário (taxas de mercado, pós-fixado TR) | {nome_ultimo_mes}\nBase Olinda - Banco Central do Brasil",
          fontweight="bold", fontsize=11)
plt.xlabel("Instituição financeira", fontweight="bold")
plt.ylabel("Taxa de juros ao mês (%)", fontweight="bold")
plt.xticks(rotation=45, ha="right")
plt.ylim(0, grafico_barras["taxa"].max() * 1.15)
plt.tight_layout()
plt.show()

---
Gráfico de linhas: evolução mês a mês

In [ ]:
tabela_linhas = tabela_juros.pivot(index="anoMes", columns="instituicao", values="taxa")

nomes_meses = tabela_juros.drop_duplicates("anoMes").set_index("anoMes")["Mes"]
tabela_linhas.index = tabela_linhas.index.map(nomes_meses)

print("Meses sem taxa por instituição:")
print(tabela_linhas.isna().sum())

plt.style.use("ggplot")

eixo = tabela_linhas.plot(kind="line", marker="o", figsize=(12, 6), colormap="tab10")

cor_da_linha = {linha.get_label(): linha.get_color() for linha in eixo.get_lines()}

ultimos_valores = tabela_linhas.iloc[-1].dropna().sort_values(ascending=False)

distancia_minima = 0.016
altura_anterior = None
posicao_do_texto = {}
for instituicao, valor in ultimos_valores.items():
    altura = valor
    if altura_anterior is not None and altura_anterior - altura < distancia_minima:
        altura = altura_anterior - distancia_minima
    posicao_do_texto[instituicao] = altura
    altura_anterior = altura

ultima_posicao = len(tabela_linhas) - 1
for instituicao, valor in ultimos_valores.items():
    eixo.text(ultima_posicao + 0.4, posicao_do_texto[instituicao],
              f"{valor:.2f}%",
              color=cor_da_linha[instituicao],
              va="center", fontsize=8, fontweight="bold")
eixo.set_xlim(-0.5, ultima_posicao + 2.5)

plt.title("Evolução da Taxa de Juros ao Mês - Financiamento Imobiliário (taxas de mercado, pós-fixado TR)\nBase Olinda - Banco Central do Brasil",
          fontweight="bold", fontsize=11)
plt.xlabel("Mês", fontweight="bold")
plt.ylabel("Taxa de juros ao mês (%)", fontweight="bold")
plt.xticks(range(len(tabela_linhas)), tabela_linhas.index, rotation=90, fontsize=8)
plt.legend(title="Instituição", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig("/content/grafico_taxa_juros_linhas.png", bbox_inches="tight")
plt.show()

display(tabela_linhas.round(2))

---
# Apostas (Bets): depósitos e apostadores por mês

**O que este notebook faz**
1. Baixa do endpoint do grupo (mockapi) os dados mensais do Ministério da Fazenda: quanto foi depositado nas bets e quantos apostadores teve.
2. Arruma os dados (nome das colunas, meses, unidades).
3. Mostra a evolução mês a mês em dois gráficos.

**De onde vêm os dados**

Não existem bases abertas sobre apostas. Por isso o grupo copiou os dados do Ministério da Fazenda pra um endpoint (mockapi), que devolve um JSON. Aqui a gente faz um GET nesse endereço.

---
Configurações das apostas

In [ ]:
url_bets = "https://6aa1d1d4ccb3db9689a5cff3.mockapi.io/endividamento-mf-2026"

---
Download dos dados

In [ ]:
resposta_bets = requests.get(url_bets, timeout=30)
resposta_bets.raise_for_status()

registros_bets = resposta_bets.json()

tabela_bets_bruta = pd.DataFrame(registros_bets)
print("Formato da tabela baixada (linhas, colunas):", tabela_bets_bruta.shape)
display(tabela_bets_bruta)

---
Tratamento dos dados

In [ ]:
tabela_bets = tabela_bets_bruta.copy()

tabela_bets = tabela_bets.rename(columns={"Numero-apostadores": "apostadores", "Deposito-mes": "deposito"})

tabela_bets["id"] = tabela_bets["id"].astype(int)
tabela_bets["apostadores"] = pd.to_numeric(tabela_bets["apostadores"], errors="coerce")
tabela_bets["deposito"] = pd.to_numeric(tabela_bets["deposito"], errors="coerce")
tabela_bets = tabela_bets.dropna(subset=["apostadores", "deposito"])
tabela_bets = tabela_bets.sort_values("id").reset_index(drop=True)

tabela_bets["data"] = pd.to_datetime(tabela_bets["Mes"], format="%m/%Y")
tabela_bets = tabela_bets.sort_values("data").reset_index(drop=True)

if tabela_bets["data"].duplicated().any():
    print("ATENÇÃO: ainda existe mês repetido na tabela.")
todos_os_meses = pd.date_range(tabela_bets["data"].min(), tabela_bets["data"].max(), freq="MS")
faltando = todos_os_meses.difference(pd.DatetimeIndex(tabela_bets["data"]))
if len(faltando) > 0:
    print("ATENÇÃO: faltam estes meses:", [mes.strftime("%m/%Y") for mes in faltando])

tabela_bets["mes"] = tabela_bets["data"].dt.strftime("%m/%Y")
tabela_bets["deposito_bi"] = tabela_bets["deposito"] / 1_000_000_000
tabela_bets["apostadores_mi"] = tabela_bets["apostadores"] / 1_000_000

print("Meses na tabela:", len(tabela_bets))
print("Período:", tabela_bets["mes"].iloc[0], "até", tabela_bets["mes"].iloc[-1])
display(tabela_bets[["mes", "deposito_bi", "apostadores_mi"]].round(2))

---
Gráficos: evolução mês a mês

In [ ]:
plt.style.use("ggplot")

tabela_grafico = tabela_bets.set_index("mes")[["deposito_bi", "apostadores_mi"]]

eixos = tabela_grafico.plot(kind="line", marker="o", subplots=True, sharex=True,
                            figsize=(12, 8), legend=False, color=["#4f46e5", "#dc2626"])

for posicao, valor in enumerate(tabela_grafico["deposito_bi"]):
    eixos[0].annotate(f"{valor:.1f}", (posicao, valor), textcoords="offset points",
                      xytext=(0, 8), ha="center", fontsize=8)
for posicao, valor in enumerate(tabela_grafico["apostadores_mi"]):
    eixos[1].annotate(f"{valor:.1f}", (posicao, valor), textcoords="offset points",
                      xytext=(0, 8), ha="center", fontsize=8)

eixos[0].set_title("Depósitos nas bets por mês (R$ bilhões)", fontweight="bold")
eixos[0].set_ylabel("R$ bilhões", fontweight="bold")
eixos[0].set_ylim(0, tabela_grafico["deposito_bi"].max() * 1.15)
eixos[1].set_title("Número de apostadores por mês (milhões de pessoas)", fontweight="bold")
eixos[1].set_ylabel("Milhões de pessoas", fontweight="bold")
eixos[1].set_ylim(0, tabela_grafico["apostadores_mi"].max() * 1.15)
eixos[1].set_xlabel("Mês", fontweight="bold")

plt.suptitle(f"Apostas no Brasil: evolução mensal ({tabela_bets['mes'].iloc[0]} a {tabela_bets['mes'].iloc[-1]})\nFonte: Ministério da Fazenda (dados do endpoint do grupo)",
             fontweight="bold", fontsize=12)
plt.xticks(range(len(tabela_grafico)), tabela_grafico.index, rotation=45, ha="right")
plt.tight_layout()
plt.savefig("/content/grafico_bets_evolucao.png", bbox_inches="tight")
plt.show()

---
# Dados do IBGE: população e renda por estado

**O que este notebook faz**
1. Baixa do endpoint (mockapi) os dados do IBGE por estado: população, renda per capita, IDH, área e outros.
2. Deixa a pessoa escolher o estado num menu e clicar em "Analisar" pra ver a evolução da população e da renda per capita.

**De onde vêm os dados**

Do endpoint (mockapi) montado com dados do IBGE.

**Limites conhecidos**
- A renda per capita de 2026 não existe ainda (o IBGE não divulgou), por isso o gráfico vai de 2022 a 2025.
- A população de 2023 é igual à de 2022 (o IBGE não publicou estimativa de 2023). Por isso o gráfico marca "2023*".
- O IBGE calculou a renda de 2022 com a 5ª visita da Pesquisa PNAD e a de 2023 em diante com a 1ª visita. Então comparar 2022 com 2023 não é perfeito.

---
Bibliotecas do menu e do botão

In [ ]:
import matplotlib.ticker as ticker
import ipywidgets as widgets
from IPython.display import display, clear_output

---
Configurações do IBGE

In [ ]:
url_mockapi = "https://6aa1d1d4ccb3db9689a5cff3.mockapi.io/densidade-ibge-2026"

---
Download dos dados

In [ ]:
print("Fazendo requisição HTTP para o endpoint JSON...")

res = requests.get(url_mockapi, timeout=30)

if res.status_code == 200:
    dados_json = res.json()
    df_geral = pd.DataFrame(dados_json)

    print(f"Total de estados encontrados: {len(df_geral)}")
    print(f"Estados disponíveis: {', '.join(df_geral['UF'].unique())}\n")

else:
    print(
        f"Erro ao acessar a API. Código: {res.status_code}"
    )

---
Gráfico: população e renda por estado (menu e botão)

In [ ]:
seletor_estado = widgets.Dropdown(
    options = sorted(df_geral["UF"].unique()),
    description = "Estado:",
    style = {"description_width": "initial"}
)

pesquisar = widgets.Button(
    description="Analisar",
    button_style = "success",
    icon="bar-chart"
)

saida = widgets.Output()

def gerar_analise(b):
    with saida:
        clear_output(wait=True)
        uf_pesquisa = seletor_estado.value
        df_uf = df_geral[df_geral['UF'].str.casefold() == uf_pesquisa.casefold()]

        if not df_uf.empty:
            info = df_uf.iloc[0]
            print(f"\nEstado: {info['UF']}")
            print(f"Área territorial: {info.get('Area-territorial')} km²")
            print(f"IDH (2021): {info.get('IDH-2021')}")
            print(f"Densidade demográfica (2022): {info.get('Densidade-demografica-2022')} hab/km²")

            anos = ['2022', '2023', '2024', '2025']

            populacao_milhoes = [info[f'Populacao-estado-{ano}'] / 1e6 for ano in anos]
            renda_per_capita = [info[f'Renda-per-capta-{ano}'] for ano in anos]

            rotulo_anos = [ano if ano != '2023' else '2023*' for ano in anos]

            df_serie = pd.DataFrame({
                'Ano': rotulo_anos,
                'Populacao_Milhoes': populacao_milhoes,
                'Renda_Per_Capta': renda_per_capita
            })

            print("\nEVOLUÇÃO DEMOGRÁFICA E ECONÔMICA:")
            print(df_serie.round(2))
            print("* 2023: no endpoint a população é igual à de 2022 (não há valor próprio pra 2023).")

            fig, ax1 = plt.subplots(figsize=(10, 5))

            df_serie.set_index('Ano')[['Renda_Per_Capta', 'Populacao_Milhoes']].plot(
                ax=ax1, secondary_y='Populacao_Milhoes', color=['#1f77b4', '#e67e22'],
                style=['-', '--'], linewidth=3, legend=False)
            ax2 = ax1.right_ax

            color = '#1f77b4'
            ax1.set_xlabel('Ano de Referência', fontweight='bold', fontsize=11)
            ax1.set_ylabel('Renda Média Per Capita (R$)', color=color, fontweight='bold', fontsize=11)
            ax1.tick_params(axis='y', labelcolor=color)
            ax1.yaxis.set_major_formatter(ticker.StrMethodFormatter('R$ {x:,.0f}'))

            color = '#e67e22'
            ax2.set_ylabel('População Total (Milhões de Hab.)', color=color, fontweight='bold', fontsize=11)
            ax2.tick_params(axis='y', labelcolor=color)

            line1 = ax1.get_lines()
            line2 = ax2.get_lines()
            line1[0].set_label('Renda Per Capita (R$)')
            line2[0].set_label('População (Milhões)')

            plt.title(f'Potencial Financeiro e Demográfico - {info["UF"]}\n(Subsídio para Análise de Crédito Regional)', fontsize=13, fontweight='bold', pad=15)

            ax1.grid(axis='y', linestyle='-', linewidth=0.7, color='#e0e0e0')
            ax1.spines['top'].set_visible(False)
            ax2.spines['top'].set_visible(False)

            lines = line1 + line2
            labels = [l.get_label() for l in lines]
            ax1.legend(lines, labels, loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False, fontsize=10)

            plt.tight_layout()
            plt.show()
        else:
            print(f"Estado '{uf_pesquisa}' não encontrado")

pesquisar.on_click(gerar_analise)
display(widgets.HBox([seletor_estado, pesquisar]), saida)